In [11]:
import numpy as np
import torch
from torch.linalg import solve_triangular
import opt_einsum

from ngo.ml.basisfunctions import GaussianRBFBasis

ImportError: cannot import name 'GaussianRBFBasis' from 'ngo.ml.basisfunctions' (C:\Users\s145608\git\ngo-public\src\ngo\ml\basisfunctions.py)

In [2]:
class GRF:
    def __init__(self, N_samples, l, device, dtype):
        super().__init__()
        self.device = device
        self.dtype = dtype
        self.N_samples = N_samples #Number of GRF samples
        self.l = torch.tensor(l, device=self.device, dtype=dtype) #Length scale of GRF
        self.d = len(l) #Dimensionality of GRF
        self.l_Dx_ratio = 2
        self.n = tuple((self.l_Dx_ratio*(1.0 / self.l).ceil().long() + 1).tolist())
        print(self.n)
        self.mus = self.get_mus()
        self.coeffs = self.compute_coeffs()
        
    def get_mus(self):
        mus = ()
        for i in range(self.d):
            mus += (torch.linspace(0,1, self.n[i], device=self.device, dtype=self.dtype),)
        return mus

    def compute_covariances(self):
        K_i = ()
        for i in range(self.d):
            scaled_mus = self.mus[i] / self.l[i]
            diff = scaled_mus[:, None] - scaled_mus[None, :]
            dist_sq = diff**2
            K_i += (torch.exp(-0.5 * dist_sq),)
        return K_i
    
    def compute_coeffs(self):
        K_i = self.compute_covariances()
        L_i = tuple(torch.linalg.cholesky(K) for K in K_i)
        Z = torch.randn(size=(self.N_samples,*self.n), device=self.device, dtype=self.dtype)
        f = Z
        for i in range(self.d):
            f = torch.tensordot(f, L_i[i].T, dims=([1],[0]))
        f = torch.tensor(f, device=self.device, dtype=self.dtype)
        # f_1 = f.reshape(self.N_samples, torch.prod(torch.tensor(self.n)).item())
        # K = torch.kron(K_i[0], torch.kron(K_i[1], K_i[2]))
        # K_full = torch.kron(K_i[0], torch.kron(K_i[1], K_i[2]))
        # K_sep = K_i[0]
        # for i in (range(1,self.d)):
        #     K_sep = torch.kron(K_sep, K_i[i])
        # print(torch.allclose(K_full, K_sep))
        # L_i = tuple(torch.linalg.cholesky(K) for K in K_i)
        # L = torch.kron(L_i[0], torch.kron(L_i[1], L_i[2]))
        # z = Z.reshape(self.N_samples, torch.prod(torch.tensor(self.n)).item())
        # f_2 = opt_einsum.contract('ij,Nj->Ni', L, z)
        # print(f_1 - f_2)
        y = f
        # Forward solves
        for i in range(self.d):
            y = torch.moveaxis(y, 1, -1)
            y = torch.linalg.solve_triangular(L_i[i].T, y, upper=True, left=False)
        #Backward solves
        f_m = y
        for i in range(self.d):
            f_m = torch.moveaxis(f_m, 1, -1)
            f_m = torch.linalg.solve_triangular(L_i[i], f_m, upper=False, left=False)
        # f_m_1 = f_m.reshape(self.N_samples, -1)
        # L_inv_i = tuple(torch.linalg.inv(L) for L in L_i)
        # L_inv = torch.kron(L_inv_i[0], L_inv_i[1])
        # L_inv = torch.kron(L_inv, L_inv_i[2])
        # cov_inv = opt_einsum.contract('ij,jk->ik', L_inv.T, L_inv)
        # f_hat = opt_einsum.contract('ij,Nj->Ni', cov_inv, f.reshape(self.N_samples, -1))
        # f_m_2 = f_hat
        # print(f_m_1 - f_m_2)
        return f_m


In [3]:
torch.cuda.empty_cache()
lambdas = (0.5,0.1,0.1)
grf = GRF(N_samples=10, l=lambdas, device='cpu', dtype=torch.float64)

(5, 21, 21)


C:\Users\s145608\AppData\Local\Temp\ipykernel_6240\2269198007.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  f = torch.tensor(f, device=self.device, dtype=self.dtype)


In [4]:
grf.coeffs.shape

torch.Size([10, 5, 21, 21])

In [5]:
grf.mus[0]

tensor([0.0000, 0.2500, 0.5000, 0.7500, 1.0000], dtype=torch.float64)

In [6]:
bases = ()
for i in range(len(lambdas)):
    bases += (GaussianRBFBasis(sigmas=int(grf.l[i]),mus=grf.mus[i]),)

NameError: name 'GaussianRBFBasis' is not defined

In [ ]:
bases[0].sigmas

0